# Agentic RAG with a Document Retrieval Tool

**Course author:** Jayneel Girnara

Extend a basic RAG workflow into an agent: the model can decide to call a retrieval tool, use the returned document excerpts, and formulate an answer.

## Learning goals
- Load, split, and index a PDF in an in-memory vector store.
- Expose similarity search as a typed agent tool.
- Create an agent with an explicit grounding instruction.
- Inspect the agent's messages and answer.

## Before you start
Install dependencies, place `medical_report.pdf` under `data/`, and configure `OPENAI_API_KEY` locally in .env. Model and embedding calls may incur charges. Do not use this example for real medical decisions; it is a learning demo, not clinical advice.

## How to work through it
Run cells in order. First build the index, then define and test the retrieval tool, create the agent, and ask a question supported by the PDF. Check retrieved evidence before trusting an answer.

## 1. Prepare and index the PDF

This setup uses the same document → chunks → embeddings → vector-store foundation as basic RAG. The in-memory store is temporary and must be rebuilt after a kernel restart. Embeddings are created by a hosted model, so the source text is sent to that provider.

**Checkpoint:** What work must be repeated after restarting the kernel?

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "data").is_dir():
    project_root = project_root.parent

load_dotenv(project_root / ".env")


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

In [ ]:
loader=PyPDFLoader(project_root / "data" / "medical_report.pdf")
docs=loader.load()
len(docs)

In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splited_docs=splitter.split_documents(docs)
len(splited_docs)

In [ ]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
vector_store=InMemoryVectorStore.from_documents(
    documents=splited_docs,
    embedding=embeddings
)

## 2. Expose retrieval as an agent tool

The agent can call this search function when it needs evidence from the indexed PDF.

The retrieval function searches the vector store and returns the text of matching passages. Wrapping it as a tool lets the model request a search using a question. Returning all selected passages matters: discarding all but the last passage can remove the evidence needed for an answer.

**Checkpoint:** What should the tool return to make its answer verifiable?

In [ ]:
@tool
def retriver_tool(query:str):
    """this tool can help you to retrive relevant data of the PDF documents, and this pdf documents have 
    details about medical reports"""
    docs=vector_store.similarity_search(query=query,k=4)
    context=""
    for doc in docs:
        context += doc.page_content + "\n\n"
    return context

## 3. Configure the model and agent

The system prompt explains when to retrieve and how to treat the returned excerpts. `create_agent` provides the model with the retrieval tool; the agent may then alternate between model reasoning and tool calls. This is more flexible than a fixed chain, but less predictable.

**Checkpoint:** What is one safety benefit of restricting the agent to a retrieval-only tool?

In [ ]:
llm=ChatOpenAI(model="gpt-4o-mini")

In [ ]:
System_Prompt="""
        You are helpful assistant that answers questions using retrived context.
        Always use the 'retriver_tool' tool for questions requiring external knowledge
"""

In [ ]:
agent=create_agent(
    model=llm,
    tools=[retriver_tool],
    system_prompt=System_Prompt,
)

## 4. Ask a question grounded in the document

Inspect the message sequence as well as the final answer to confirm retrieval occurred. Verify claims against the PDF excerpts; this educational example is not medical advice and must not be used for diagnosis or treatment.

**Checkpoint:** What would make an answer unsupported even if the agent called the retriever?

In [ ]:
query="What is the name of patient and what is name of Doctors"
response=agent.invoke({"messages":[{"role":"user","content":query}]})
 

In [ ]:
result=response["messages"]
print(result)